# Known-k Temporal-GCN: фиксированные повторы 7027/7028
Тот же train/validation и та же архитектура, что у pilot 7026. Результаты остаются исследовательскими. Выполняй ячейки по одной; каждый seed требует отдельного ввода. Старые результаты, test и independent holdout не изменяются.

In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys

if importlib.util.find_spec('google.colab') is not None:
    if any(name == 'diffusion_sources' or name.startswith('diffusion_sources.') or name == 'scripts.known_k_temporal_repeats' for name in sys.modules):
        raise RuntimeError('Код проекта уже импортирован: перезапусти среду Colab перед setup')
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/diffusion-sources-known-k-repeats')
    REMOTE = 'https://github.com/1habibi/diffusion-sources-localization-.git'
    BRANCH = 'experiment/known-k-temporal-gcn-repeats'
    if not PROJECT.exists():
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REMOTE, str(PROJECT)], check=True)
    else:
        if not (PROJECT / '.git').exists():
            raise RuntimeError(f'{PROJECT} существует, но не является Git-клоном')
        clean = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True).strip()
        branch = subprocess.check_output(['git', '-C', str(PROJECT), 'branch', '--show-current'], text=True).strip()
        if clean or branch != BRANCH:
            raise RuntimeError('Клон изменён или открыта другая ветка; ничего не перезаписываем')
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
    packages = {'numpy': 'numpy', 'networkx': 'networkx', 'scipy': 'scipy', 'sklearn': 'scikit-learn', 'yaml': 'PyYAML', 'tqdm': 'tqdm', 'torch_geometric': 'torch-geometric'}
    missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT), '--no-deps'], check=True)
else:
    PROJECT = next((base for base in [Path.cwd(), *Path.cwd().parents] if (base / 'scripts/known_k_temporal_repeats.py').is_file()), None)
    if PROJECT is None:
        raise RuntimeError('Локальный проект не найден')
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT / 'src'))
import torch
from diffusion_sources.known_k_temporal_pilot import KnownKPaths
from diffusion_sources.temporal_pilot_artifacts import read_stage
from scripts import known_k_temporal_repeats as REPEATS
from scripts.known_k_temporal_repeats import RepeatPaths, preflight, run_repeat_stage
if not Path(REPEATS.__file__).resolve().is_relative_to(PROJECT.resolve()):
    raise RuntimeError('Kernel импортировал другую копию runner; перезапусти среду')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
APPROVED_7027 = False
APPROVED_7028 = False
revision = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Project:', PROJECT, 'revision:', revision)
print('Runner:', REPEATS.__file__, 'device:', DEVICE)
print('Setup не запускал freeze или обучение. Дальше — paths.')


In [ ]:
if importlib.util.find_spec('google.colab') is None:
    raise RuntimeError('Эта ячейка предназначена для Colab с подключённым Drive')
ROOT = Path('/content/drive/MyDrive/diffusion-sources')
OLD = KnownKPaths(
    data_dir=ROOT / 'data/facebook_main',
    frozen_s1b_dir=ROOT / 'reports/runs/facebook_main_v2/s1b/seed_7026',
    output_dir=ROOT / 'reports/runs/known_k_temporal_gcn_pilot/v1',
    config_path=PROJECT / 'configs/known_k_temporal_gcn_pilot.yaml',
)
PATHS = RepeatPaths(
    pilot=OLD,
    output_dir=ROOT / 'reports/runs/known_k_temporal_gcn_repeats/v1',
    protocol_path=PROJECT / 'configs/known_k_temporal_gcn_repeats.yaml',
    notebook_path=PROJECT / 'notebooks/colab_known_k_temporal_gcn_repeats.ipynb',
)
for path in (OLD.data_dir, OLD.frozen_s1b_dir, OLD.output_dir, OLD.config_path, PATHS.protocol_path, PATHS.notebook_path):
    if not path.exists():
        raise FileNotFoundError(path)
print('Old pilot (read-only):', OLD.output_dir)
print('New repeat output:', PATHS.output_dir)
print('Data:', OLD.data_dir, 'frozen S1b:', OLD.frozen_s1b_dir)
print('Пути проверены. Дальше — read-only preflight; обучения ещё нет.')


In [ ]:
CHECK = preflight(PATHS)
print('Pilot 7026 gate:', CHECK['pilot']['quality_gate'])
print('Pilot checkpoint SHA-256:', CHECK['identity']['old_pilot_checkpoint_hash'])
print('Package source files authenticated:', len(CHECK['identity']['package_hashes']))
print('Preflight только прочитал и сверил файлы. Дальше — freeze.')


In [ ]:
FROZEN = run_repeat_stage('freeze', PATHS, torch.device('cpu'))
print('Freeze:', {key: FROZEN[key] for key in ('status', 'reference_seed', 'repeat_seeds')})
print('New artifact:', PATHS.output_dir / 'freeze')
print('STOP: пришли вывод freeze. Seed 7027 пока не запускай.')


In [ ]:
APPROVED_7027 = False
if not (PATHS.output_dir / 'freeze' / 'complete').is_file():
    raise RuntimeError('Freeze не завершён; seed 7027 не подтверждён')
if DEVICE != 'cuda':
    raise RuntimeError('Для долгого обучения нужен GPU; seed 7027 не подтверждён')
answer = input('Введи RUN_KNOWN_K_REPEAT_7027 для отдельного GPU-обучения: ').strip()
if answer != 'RUN_KNOWN_K_REPEAT_7027':
    raise RuntimeError('Неверное подтверждение; seed 7027 не запущен')
APPROVED_7027 = True
print('Seed 7027 подтверждён. Следующая ячейка запускает обучение.')


In [ ]:
if globals().get('APPROVED_7027') is not True:
    raise RuntimeError('Seed 7027 не подтверждён; сначала выполни ячейку approve_7027')
APPROVED_7027 = False
REPORT_7027 = run_repeat_stage('seed_7027', PATHS, torch.device('cuda'))
pair = REPORT_7027['paired_report']
print('Seed:', REPORT_7027['seed'], 'best epoch:', REPORT_7027['best_epoch'])
print('Control F1:', pair['control']['all']['f1'], 'new F1:', pair['candidate']['all']['f1'])
print('Delta F1:', pair['delta_f1'], 'paired CI:', pair['f1_ci'])
print('Artifact:', PATHS.output_dir / 'seed_7027')
print('STOP: пришли этот короткий результат. Не запускай 7028 автоматически.')


In [ ]:
IDENTITY = preflight(PATHS)['identity']
_, VERIFIED_7027 = read_stage(PATHS.output_dir, 'seed_7027', IDENTITY)
CAN_RUN_7028 = VERIFIED_7027['delta_f1'] > 0
print('Seed 7027 delta F1:', VERIFIED_7027['delta_f1'])
if CAN_RUN_7028:
    print('Положительный повтор. После согласования можно подтвердить seed 7028.')
else:
    print('Неположительный повтор: seed 7028 запрещён. Переходи к summary.')


In [ ]:
APPROVED_7028 = False
if globals().get('CAN_RUN_7028') is not True:
    raise RuntimeError('Seed 7027 не прошёл stop-rule; 7028 запрещён')
if DEVICE != 'cuda':
    raise RuntimeError('Для долгого обучения нужен GPU; seed 7028 не подтверждён')
answer = input('Введи RUN_KNOWN_K_REPEAT_7028 для второго отдельного GPU-обучения: ').strip()
if answer != 'RUN_KNOWN_K_REPEAT_7028':
    raise RuntimeError('Неверное подтверждение; seed 7028 не запущен')
APPROVED_7028 = True
print('Seed 7028 подтверждён. Следующая ячейка запускает обучение.')


In [ ]:
if globals().get('APPROVED_7028') is not True or globals().get('CAN_RUN_7028') is not True:
    raise RuntimeError('Seed 7028 не подтверждён или stop-rule 7027 запрещает запуск')
APPROVED_7028 = False
REPORT_7028 = run_repeat_stage('seed_7028', PATHS, torch.device('cuda'))
pair = REPORT_7028['paired_report']
print('Seed:', REPORT_7028['seed'], 'best epoch:', REPORT_7028['best_epoch'])
print('Control F1:', pair['control']['all']['f1'], 'new F1:', pair['candidate']['all']['f1'])
print('Delta F1:', pair['delta_f1'], 'paired CI:', pair['f1_ci'])
print('Artifact:', PATHS.output_dir / 'seed_7028')
print('STOP: пришли этот короткий результат. Никакого independent holdout здесь нет.')


In [ ]:
SUMMARY = run_repeat_stage('summary', PATHS, torch.device('cpu'))
print('Status:', SUMMARY['status'])
print('Pilot 7026 delta:', SUMMARY['reference_7026']['delta_f1'])
print('Repeat deltas:', {seed: row['delta_f1'] for seed, row in SUMMARY['repeat_seeds'].items()})
print('Repeat mean:', SUMMARY['mean_repeat_delta_f1'], 'sample SD:', SUMMARY['sample_sd_repeat_delta_f1'])
print('Both repeats positive:', SUMMARY['positive_delta_each_repeat'])
print('Artifact:', PATHS.output_dir / 'summary')
print('STOP: пришли summary; это всё ещё exploratory validation, не independent оценка.')
